In [ ]:
import os
import pandas as pd
import re
from collections import defaultdict

In [ ]:
def process_tsv_file(file_path):
    """
    Reads a `.tsv` file, adds headers, filters rows, and removes duplicates based on `query`, 
    keeping the row with the smallest `evalue`. Assumes input is already sorted by `evalue`.

    Args:
        file_path (str): Path to the `.tsv` file.
        
    Returns:
        pd.DataFrame: Processed DataFrame containing filtered, deduplicated rows.
    """
    headers = [
        "query", "target", "fident", "alnlen", "mismatch", "gapopen",
        "qstart", "qend", "tstart", "tend", "evalue", "bits", 
        "qlen", "tlen", "qcov", "tcov", "qorfstart", "qorfend", "torfstart", "torfend"
    ]

    # Explicitly define dtypes for better performance (adjust as needed)
    dtypes = {
        "query": str, "target": str, "fident": float, "alnlen": int, 
        "mismatch": int, "gapopen": int, "qstart": int, "qend": int, 
        "tstart": int, "tend": int, "evalue": float, "bits": float, 
        "qlen": int, "tlen": int, "qcov": float, "tcov": float,
        "qorfstart": int, "qorfend": int, "torfstart": int, "torfend": int
    }
    
    try:
        # Read only necessary columns
        df = pd.read_csv(file_path, sep="\t", names=headers, header=None, dtype=dtypes)

        # Drop NaN-containing columns (likely unnecessary but kept for safety)
        df.dropna(axis=1, how="any", inplace=True)

        # Apply filtering first
        df = df[df["alnlen"] >= 100]

        # Since the file is sorted by `evalue`, dropping duplicates immediately is optimal
        df = df.drop_duplicates(subset="query", keep="first")

        return df
    except Exception as e:
        print(f"Error processing file {file_path}: {e}")
        return None

def group_files_by_sample(input_dir):
    """
    Groups TSV files by sample ID, which is defined as everything before the first period in the filename.
    For example, all files starting with "PES001." will be grouped together.

    Args:
        input_dir (str): Path to the input directory containing `.tsv` files.

    Returns:
        dict: Dictionary mapping sample IDs to lists of file paths.
    """
    file_groups = defaultdict(list)
    
    for filename in os.listdir(input_dir):
        if filename.endswith(".tsv"):
            # Extract the sample id as the part before the first period.
            sample_id = filename.split(".")[0]
            file_groups[sample_id].append(os.path.join(input_dir, filename))
    
    return file_groups

def merge_grouped_results(file_group, output_dir):
    """
    Merges processed results from different assemblers of the same dataset,
    ordering the results for each target such that:
      - Megahit hits come first,
      - followed by CarpeDeam Safe hits,
      - and then CarpeDeam Unsafe hits.
    
    Args:
        file_group (list): List of file paths for a single dataset.
        output_dir (str): Path to save the merged output files.
    """
    # Determine the output file path
    dataset_id = os.path.basename(file_group[0]).split("raw-derep")[0].strip(".")
    output_path = os.path.join(output_dir, f"{dataset_id}_merged.tsv")

    # Skip processing if merged file already exists
    if os.path.exists(output_path):
        print(f"Skipping {dataset_id} (merged file already exists: {output_path})")
        return

    assembler_data = {}  # Store processed DataFrames per assembler
    target_to_query_map = defaultdict(dict)  # Store query mappings

    # Process each assembler's TSV file
    for file_path in file_group:
        # Determine assembler name based on file_path substring
        assembler_name = "unknown"
        if "megahit" in file_path:
            assembler_name = "megahit"
        elif "configSafe" in file_path:
            assembler_name = "configSafe"
        elif "configUnsafe" in file_path:
            assembler_name = "configUnsafe"
        
        df = process_tsv_file(file_path)
        
        # Drp duplicates --> based on target or query??
        # df.drop_duplicates(subset="query", keep="first", inplace=True)
        df.drop_duplicates(subset="target", keep="first", inplace=True)
        
        if df is not None:
            # Filter rows based on allowed_queries
            # df = df[df["query"].isin(allowed_queries)]
            # Skip if DataFrame is empty after filtering
            if df.empty:
                print(f"Skipping {assembler_name} file {file_path} because it is empty after filtering.")
                continue
            # Add a temporary column to record assembler info
            df["assembler"] = assembler_name
            assembler_data[assembler_name] = df  # Store filtered DataFrame

            # Update query mappings per target
            target_to_query_map.update(df.set_index("target")["query"].to_dict())

    # Merge results while preserving all rows
    all_dfs = []
    if "megahit" in assembler_data:
        all_dfs.append(assembler_data["megahit"])
    if "configSafe" in assembler_data:
        all_dfs.append(assembler_data["configSafe"])
    if "configUnsafe" in assembler_data:
        all_dfs.append(assembler_data["configUnsafe"])
        
    if not all_dfs:
        print(f"No valid data for {dataset_id}. Skipping merge.")
        return

    merged_df = pd.concat(all_dfs, ignore_index=True)

    # Sort by target first, then by assembler priority.
    assembler_priority = {"megahit": 1, "configSafe": 2, "configUnsafe": 3}
    merged_df["priority"] = merged_df["assembler"].map(assembler_priority)
    merged_df.sort_values(by=["target", "priority"], inplace=True)
    merged_df.drop(columns=["priority"], inplace=True)

    # Optionally remove the 'assembler' column from final output
    #merged_df.drop(columns=["assembler"], inplace=True)

    # Save merged results
    merged_df.to_csv(output_path, sep="\t", index=False)
    print(f"Saved merged results: {output_path}")

    # Create mapping file – optimized
    # For each assembler, only create the mapping if the DataFrame exists and is not empty
    configSafe_map = (
        assembler_data.get("configSafe", pd.DataFrame())
        .set_index("target")["query"].to_dict()
        if "configSafe" in assembler_data else {}
    )
    configUnsafe_map = (
        assembler_data.get("configUnsafe", pd.DataFrame())
        .set_index("target")["query"].to_dict()
        if "configUnsafe" in assembler_data else {}
    )
    megahit_map = (
        assembler_data.get("megahit", pd.DataFrame())
        .set_index("target")["query"].to_dict()
        if "megahit" in assembler_data else {}
    )

    mapping_list = [
        [target, configSafe_map.get(target, "N/A"), configUnsafe_map.get(target, "N/A"), megahit_map.get(target, "N/A")]
        for target in target_to_query_map.keys()
    ]
    mapping_df = pd.DataFrame(mapping_list, columns=["target", "query_configSafe", "query_configUnsafe", "query_megahit"])
    mapping_output_path = os.path.join(output_dir, f"{dataset_id}_mapping.tsv.gz")
    mapping_df.to_csv(mapping_output_path, sep="\t", index=False, compression="gzip")
    print(f"Saved compressed mapping file: {mapping_output_path}")

    

def process_tsv_directory(input_dir, output_dir):
    """
    Processes and merges `.tsv` files by dataset groups.

    Args:
        input_dir (str): Path to the input directory containing `.tsv` files.
        output_dir (str): Path to the output directory to save processed files.
    """
    os.makedirs(output_dir, exist_ok=True)

    # Group files by dataset ID
    file_groups = group_files_by_sample(input_dir)

    # Process and merge each group
    for dataset_id, file_group in file_groups.items():
        print(f"Processing {dataset_id}") 
        merge_grouped_results(file_group, output_dir)


In [ ]:
# Function to remove "_nXXX" suffix from SampleName
def clean_sample_name(sample_name):
    return re.sub(r"_n\d+$", "", sample_name)  # Remove _nXXX at the end

In [ ]:
dir_in="/project3/data/results/assembly-prokka-mmseqs-nr"
dir_out="/project3/data/results/assembly-prokka-mmseqs-nr/merged"

In [ ]:
process_tsv_directory(dir_in, dir_out)